In [29]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.impute import SimpleImputer

data = pd.read_csv("archive/kidney_disease.csv",na_values=['?','?','?'])

for col in data.select_dtypes(include=['object']).columns:
    data[col] = data[col].astype(str).str.replace(r'\t', '', regex=True).str.strip().str.lower()
    data[col] = data[col].replace('nan',np.nan)

if 'classification' in data.columns:
    data['classification'] = data['classification'].replace({'ckd':1,'notckd':0})
    data['classification'] = pd.to_numeric(data['classification'], errors='coerce')
data = data.dropna(subset=['classification'])
data = data.drop(columns=['id'])
X = data.drop(columns=['classification'])
y = data['classification']

for col in X.select_dtypes(include=['object']).columns:
    uniqvals = X[col].dropna().unique()
    if len(uniqvals) <= 2:
        mapp = {val:i for i,val in enumerate(uniqvals)}
        X[col] = X[col].map(mapp)
X = pd.get_dummies(X, drop_first=True)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

num_cols = X_train.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train[num_cols]),columns=num_cols)
X_test_imputed = pd.DataFrame(imputer.transform(X_test[num_cols]),columns=num_cols)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_imputed)
X_test = scaler.transform(X_test_imputed)

pca = PCA(n_components=0.95,random_state=42)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)



model = LogisticRegression(max_iter=1000)
model.fit(X_train_pca,y_train)

y_pred = model.predict(X_test_pca)
print(f"Accuracy Test {accuracy_score(y_test,y_pred):.4f}")
print("classification Report -> ")
print(classification_report(y_test,y_pred))






Accuracy Test 0.9875
classification Report -> 
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        30
           1       1.00      0.98      0.99        50

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80



C:\Users\SANJAYVARAPRASAD\AppData\Local\Temp\ipykernel_31880\1738955656.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in data.select_dtypes(include=['object']).columns:
C:\Users\SANJAYVARAPRASAD\AppData\Local\Temp\ipykernel_31880\1738955656.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas